# Session 7 · Text II: Meaning Over Words

**Goal:** search notes by meaning, classify them from a handful of examples, discover what officers write about without being told, and compare it all with session 6's hand-built rules.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 7

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import numpy as np
import pandas as pd
from lnl import pipeline, text
pd.set_option("display.max_colwidth", 120)

df = pipeline.clean_through(4)
notes = text.sample_notes(df)                            # the same 10,000 notes as session 6
tokens = text.tokenize(notes["note"], text.load_spacy())
rules = text.best_rules(notes["note"], tokens)          # session 6's finished rules, for comparison

## 1. Embeddings: meaning as coordinates
An **embedding** turns a piece of text into a list of numbers, a point on a map of meaning. With a good model, notes that mean similar things land near each other even when they share few words.

The first run downloads a small model (about 90 MB), then turns 10,000 notes into embeddings. Allow a minute or two on a laptop-class CPU. If the network blocks the download, the notebook says so and falls back to an older technique, Latent Semantic Analysis. Everything still runs, but the results are noticeably weaker, which is its own lesson about model choice. (Instructors: `EMBEDDING_MODEL` in `lnl/config.py` can point at a copy of the model in a volume.)

In [ ]:
embedder = text.Embedder()
vectors = embedder.fit_encode(notes["note"])
print(vectors.shape, "(notes x numbers per note)")

In [ ]:
demo = ["Inmate was squared away.", "No problems today.", "Inmate refused orders.", "Good time credit applied."]
demo_vectors = embedder.encode(demo)
pd.DataFrame(demo_vectors @ demo_vectors.T, index=demo, columns=demo).round(2)   # 1.0 = same meaning

## 2. Semantic search
Describe what you want in plain English, and get the notes closest in meaning:

In [ ]:
text.semantic_search(embedder, vectors, notes, "the inmate behaved well and followed orders")

Look closely at what comes back. Embeddings are famously weak at **negation**: *not a good day* can land right next to *behaved well*, because both are about the same topic. That's why nothing here gets trusted until it's scored.

🧪 **Your turn #1.** Write your own query and see what comes back. Try something no note says word for word.

In [ ]:
MY_QUERY = "inmate was violent toward staff"      # change this and run again
text.semantic_search(embedder, vectors, notes, MY_QUERY)

## 3. Classify with examples instead of rules (zero-shot)
Instead of listing every good word, give a few example sentences of what *good* looks like and what *not good* looks like. Each note gets whichever group it's closest to.

In [ ]:
GOOD_EXAMPLES = ["The inmate behaved well and followed all instructions.",
                 "No problems, cooperative and respectful.",
                 "Excellent conduct this week."]
OTHER_EXAMPLES = ["The inmate was combative and refused orders.",
                  "Routine paperwork, nothing to report.",
                  "Behavior was not good today."]

closeness_to_good = (vectors @ embedder.encode(GOOD_EXAMPLES).T).max(axis=1)
closeness_to_other = (vectors @ embedder.encode(OTHER_EXAMPLES).T).max(axis=1)
zero_shot = closeness_to_good > closeness_to_other

## 4. Learn from a few hundred labeled notes (few-shot)
In real life, a supervisor labels a few hundred notes, about an hour of work. Here the answer key stands in for that supervisor. The model learns from 300 notes and is tested on the other 9,700, which it never saw.

In [ ]:
from sklearn.linear_model import LogisticRegression

TRAIN_SIZE = 300                    # 🧪 Your turn #2: try 50, then 1000, and rerun this cell and the next
order = np.random.RandomState(0).permutation(len(notes))
train, test = order[:TRAIN_SIZE], order[TRAIN_SIZE:]

model = LogisticRegression(max_iter=1000).fit(vectors[train], notes["is_good"].values[train])
few_shot = model.predict(vectors[test])

In [ ]:
board = text.Scoreboard(notes["is_good"].values[test])     # everything scored on the same held-out notes
board.add("Session 6 rules (7 rounds of hand-tuning)", rules[test])
board.add("Zero-shot (6 example sentences)", zero_shot[test])
board.add(f"Few-shot ({TRAIN_SIZE} labeled notes)", few_shot)
display(board.show())
board.plot();

Zero-shot depends heavily on the model: respectable with a sentence-transformer, weak with the fallback. Few-shot is much less sensitive to that, and a few hundred labeled examples typically match or beat hours of hand-written rules. The approach also carries over to the next question (*"find notes about medical issues"*) without starting over. Rules stay useful where you must be able to explain exactly why a record was selected.

## 5. Topic modeling: what are officers writing about?
Nobody tells a topic model what to look for. It groups notes that use similar words, which is useful when you don't yet know what questions to ask.

In [ ]:
topics, topic_of_note = text.topic_table(notes["note"], n_topics=6)
topics

Several "topics" are just boilerplate: *shift*, *logged*, *cape stored in property*, *will continue to monitor*. Real narrative data does this too. The fix is a **domain stop-word list**, words that appear everywhere and mean nothing for this question.

🧪 **Your turn #3.** Run with the stop lists below, then try `N_TOPICS = 4` and `10`.

In [ ]:
BOILERPLATE = ["shift", "day", "2nd", "3rd", "obs", "count", "note", "logged", "sgt", "reports", "monitor",
               "continue", "powers", "use", "observed", "cape", "stored", "property", "inmate", "offender", "subject"]
OFFICER_NAMES = ["ramirez", "lee", "goode", "goodman", "nguyen", "patel", "brooks", "davis", "okafor", "castillo"]
N_TOPICS = 6

topics, topic_of_note = text.topic_table(notes["note"], n_topics=N_TOPICS, extra_stop_words=BOILERPLATE + OFFICER_NAMES)
topics

In [ ]:
examples = notes.assign(topic=topic_of_note).drop_duplicates("note")
examples.groupby("topic").head(3).sort_values("topic")[["topic", "note", "true_label"]]

## 6. In production on Databricks
Databricks **AI Functions** (`ai_analyze_sentiment()`, `ai_classify()`) run a hosted language model over a column straight from SQL, no Python required. Two rules before using them on real data:
1. Real conduct notes are **criminal justice information**. Sending them through any AI model goes through **AI governance review first**.
2. **Measure it** exactly the way we did here: against a labeled sample, with precision and recall, before anyone relies on it.

---
## Series wrap-up

| Session | The one thing to remember |
|---|---|
| 1 · Meet the Mess | Clean-looking answers can be wrong. Whitespace hides from `isna()`. |
| 2 · Cleansing | Load as text. "Unknown" codes are the data owner's call. Match on standardized values. |
| 3 · Dates | Explicit formats, century rules by meaning, and readable ≠ true. |
| 4 · Types | IDs are text, money is decimal, codes are categories. Report what didn't convert. |
| 5 · PySpark | Same rules, different engine. Reconcile before you trust. |
| 6 · Text I | Rules are transparent but brittle. Always score against an answer key. |
| 7 · Text II | Meaning-based methods plus a few hundred labels usually win. Governance first. |

---
## Answer key
There are no checked answers this session. For #2: accuracy climbs quickly from 50 to a few hundred labels, then levels off. That's why a few hundred labels is usually enough to start. For #3: with boilerplate removed, the topics line up with real themes: praise, refusals and fights, scheduled activities, and the sentence-credit paperwork that fooled the keyword search in session 6.